In [2]:
import sys
sys.path.append('../')
import karman
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

In [3]:
device = 'cpu'
torch_type = 'float32'
batch_size = 64
normalization_dict_path = None
model_path = None
lr = 0.001
run_name = ''
thermo_path = '/shared/us-hl-therm-landing/satellites_data_w_sw_2mln.csv'
epochs = 5
num_workers = 0
min_date = '2000-07-29 00:59:47'
max_date = '2024-05-31 23:59:32'
omni_indices_path = '/shared/us-hl-therm-landing/omniweb_data/merged_omni_indices.csv'
omni_magnetic_field_path = '/shared/us-hl-therm-landing/omniweb_data/merged_omni_magnetic_field.csv'
omni_solar_wind_path = '/shared/us-hl-therm-landing/omniweb_data/merged_omni_solar_wind.csv'
nrlmsise00_path = '/shared/us-hl-therm-landing/nrlmsise00_data/nrlmsise00_time_series.csv'
goes_path = None
soho_path = '/shared/us-hl-therm-landing/soho_data/soho_data.csv'
lag_minutes = 0
resolution_minutes = 10
dropout = 0.05
state_size = 64
lstm_layers = 2
attention_heads = 4
wandb_inactive = False
features_to_exclude_thermo = []

In [4]:
karman_dataset = karman.KarmanDataset(
    thermo_path=thermo_path,
    min_date=pd.to_datetime(min_date),
    max_date=pd.to_datetime(max_date),
    normalization_dict=None,
    nrlmsise00_path=nrlmsise00_path,
    omni_indices_path=omni_indices_path,
    omni_magnetic_field_path=omni_magnetic_field_path,
    omni_solar_wind_path=omni_solar_wind_path,
    soho_path=soho_path,
    lag_minutes_nrlmsise00=lag_minutes,
    nrlmsise00_resolution=resolution_minutes,
    lag_minutes_omni=lag_minutes,
    omni_resolution=resolution_minutes,
    lag_minutes_soho=lag_minutes,
    soho_resolution=resolution_minutes,
    # features_to_exclude_thermo=features_to_exclude_thermo
)

Loading Omni indices.


/home/jordivilaperez/2024-HL-Thermo-CL/notebooks/../karman/dataset.py:469: FutureWarning: DataFrame.interpolate with method=pad is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.time_series_data[data_name]["data"] = self.time_series_data[data_name]["data"].interpolate(method="pad")
/home/jordivilaperez/2024-HL-Thermo-CL/notebooks/../karman/dataset.py:472: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  self.time_series_data[data_name]["data"] = (self.time_series_data[data_name]["data"].resample(f"{resolution}T").ffill())


Loading Omni Solar Wind.


/home/jordivilaperez/2024-HL-Thermo-CL/notebooks/../karman/dataset.py:469: FutureWarning: DataFrame.interpolate with method=pad is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.time_series_data[data_name]["data"] = self.time_series_data[data_name]["data"].interpolate(method="pad")
/home/jordivilaperez/2024-HL-Thermo-CL/notebooks/../karman/dataset.py:472: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  self.time_series_data[data_name]["data"] = (self.time_series_data[data_name]["data"].resample(f"{resolution}T").ffill())


Loading Omni Magnetic Field.


...

Creating thermospheric density dataset
Removing from the data errors in mean absolute percentage error 200% or more in the density (between nrlmsise00 and ground truth)
loading it from file
passed mask in exclude_mask.pk does not exist or not valid, computing it


100%|██████████| 132/132 [00:01<00:00, 108.21it/s]


saving it to pickle file: exclude_mask.pk
Used features: Index(['tudelft_thermo__altitude__[m]', 'tudelft_thermo__latitude__[deg]',
       'celestrack__ap_average__',
       'space_environment_technologies__f107_obs__',
       'space_environment_technologies__f107_average__',
       'space_environment_technologies__s107_obs__',
       'space_environment_technologies__s107_average__',
       'space_environment_technologies__m107_obs__',
       'space_environment_technologies__m107_average__',
       'space_environment_technologies__y107_obs__',
       'space_environment_technologies__y107_average__', 'JB08__d_st_dt__[K]',
       'tudelft_thermo__longitude__[deg]_sin',
       'tudelft_thermo__longitude__[deg]_cos', 'all__day_of_year__[d]_sin',
       'all__day_of_year__[d]_cos', 'all__seconds_in_day__[s]_sin',
       'all__seconds_in_day__[s]_cos'],
      dtype='object')
Index(['tudelft_thermo__altitude__[m]', 'tudelft_thermo__latitude__[deg]',
       'celestrack__ap_average__',
       '

/home/jordivilaperez/2024-HL-Thermo-CL/notebooks/../karman/dataset.py:496: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  self.time_series_data[data_name]["data"] = (self.time_series_data[data_name]["data"].resample(f"{resolution}T").ffill())



Finished Creating dataset.


In [5]:
altitude_bins_classification = {'0-200 km': 0,
                                '200-250 km': 1,
                                '250-300 km': 2,
                                '300-350 km': 3,
                                '350-400 km': 4,
                                '400-450 km': 5,
                                '450-500 km': 6,
                                '500-550 km': 7,
                                '550-600 km': 8}

solar_activity_bins_classification = {'F10.7: 0-70 (low)': 0,
                                      'F10.7: 70-150 (moderate)': 1,
                                      'F10.7: 150-200 (moderate-high)': 2,
                                      'F10.7: 200 (high)': 3}

storm_types = {
    "G0": 0,
    "G1": 1,
    "G2": 2,
    "G3": 3,
    "G4": 4,
    "G5": 5,
}

In [6]:
#Train, validation, test splits:
idx_test_fold=2
test_month_idx = 2 * (idx_test_fold - 1)
validation_month_idx = test_month_idx + 2
print(test_month_idx,validation_month_idx)
karman_dataset._set_indices(test_month_idx=[test_month_idx], validation_month_idx=[validation_month_idx],custom={
    2001: {"validation":2,"test":3},
    2003: {"validation":9, "test":10},
    2005: {"validation":4, "test":5},
    2012: {"validation":8, "test":9},
    2013: {"validation":4, "test":5},
    2015: {"validation":2, "test":3},
    2022: {"validation":0, "test":1},
    2024: {"validation":3,"test":4}}
)
train_dataset = karman_dataset.train_dataset()
validation_dataset = karman_dataset.validation_dataset()
test_dataset = karman_dataset.test_dataset()

2 4
Creating training, validation and test sets.


25 years to iterate through.: 100%|██████████| 25/25 [00:03<00:00,  6.71it/s]

Train size: 1641897
Validation size: 162991
Test size: 175278


In [11]:
train_batch = torch.utils.data.DataLoader(train_dataset, batch_size=len(train_dataset))
val_batch = torch.utils.data.DataLoader(validation_dataset, batch_size=len(validation_dataset))
test_batch = torch.utils.data.DataLoader(test_dataset, batch_size=len(test_dataset))

In [31]:
sample = iter(train_batch)
print(sample()))


AttributeError: '_SingleProcessDataLoaderIter' object has no attribute 'next'

In [19]:
data_test = pd.DataFrame()
for batch_idx,el in enumerate(train_batch):
    print(1)
    # for k,v in el.items():

    #     if k in exclude:
    #         continue 

    #     # if k not in avg_meters:
    #     #     data_test[k] = AverageMeter()
        
    #     match k:
    #         case 'geomagnetic_storm_G_class':
    #             data_test.update(storm_types[v])
    #         case 'altitude_bins': 
    #             data_test[k].update(altitude_bins_classification[v])
    #         case 'solar_activity_bins':
    #             data_test[k].update(solar_activity_bins_classification[v])
    #         case _:
    #             print(v)
    #             data_test[k].update(v)

KeyboardInterrupt: 

In [ ]:
data_train = []



for i in range(len(train_dataset)):
    features, labels = train_dataset[i]
    # Convert tensors to lists or any suitable format
    features = features.tolist()
    data_train.append(features + [labels])

# Create DataFrame
columns = [f'feature_{i}' for i in range(len(data_train[0]) - 1)] + ['label']
df = pd.DataFrame(data_train, columns=columns)

print(df.head())